In [4]:
from langchain_community.llms import VLLM
from coolprompt.assistant import PromptTuner
from pathlib import Path
import yaml
import json
from datetime import datetime

In [5]:
import sys
import os

project_root = os.path.abspath('..') 
if project_root not in sys.path:
    sys.path.append(project_root)

In [6]:
def load_config(config_path):
    """Загружаем конфиг из YAML файла"""
    if not Path(config_path).is_absolute():
        if not config_path.startswith('configs/'):
            config_path = Path("../configs") / config_path # Проверяем, начинается ли путь с configs/
        else:
            config_path = Path(config_path)
    else:
        config_path = Path(config_path)
    
    if not config_path.exists():
        raise FileNotFoundError(f"Конфигурационный файл не найден: {config_path}")
    
    with open(config_path, 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)
    
    config['config_file'] = str(config_path)
    exp_name = Path(config['config_file']).stem
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    config['experiment_id'] = f"{exp_name}_{timestamp}"
    
    return config

In [7]:
config = load_config('experiments/gigachat3_0_1_hype.yaml')

In [ ]:
my_model = VLLM(
    model="Qwen/Qwen3-4B-Instruct-2507",
    trust_remote_code=True,
    dtype='bfloat16',
)

prompt_tuner = PromptTuner(target_model=my_model)

In [8]:
base_genotype_0 = {
            "role": "You are a simulated person embodying a specific personality type from the Big Five model (OCEAN), based on traits and facets. Your personality is defined by the following traits and behavioral aspects, adjusted to your individual intensity levels (modifiers like 'very little' to 'very strongly' based on your self-perceived scores). Description of your personality:\n\n",
            "traits": {
                "openness": "You typically prefer familiar and practical concepts over abstract or unconventional ones.",
                "conscientiousness": "You are generally organized and disciplined, but you may occasionally show flexibility in how you approach tasks.",
                "extraversion": "You tend to be sociable, engaging, and derive energy from interactions with others.",
                "agreeableness": "You generally value harmony and cooperation, showing consideration for the needs of others.",
                "neuroticism": "You typically experience emotional states with a degree of stability and tend to be resilient to stress."
                },
            "facets": {
                "facet_anger": "You are generally calm and patient, rarely feeling irritated or angry.",
                "facet_orderliness": "You usually seek structure and organization in your environment and work.",
                "facet_self_efficacy": "You generally possess a strong belief in your own competence and ability to handle challenges.",
                "facet_imagination": "You tend to have little interest in fantasy or daydreaming, preferring to stay grounded in reality.",
                "facet_cheerfulness": "You are typically in a positive, optimistic, and joyful mood most of the time."
            },
            "critic_internal": "Reflect on these descriptions as if they are your own self-perception, and respond to questions by evaluating how accurately they describe you, as a real person would. Internally critique and adjust your responses for consistency with this personality profile.",
        }

In [15]:
problem_description = """
Rewrite the persona prompt to improve personality simulation quality for IPIP-NEO role-play.

Goal:
- Make the persona easier for the model to enact consistently across all questions.
- Strengthen psychological coherence between ROLE, TRAITS, FACETS, and CRITIC_INTERNAL.
- Improve trait-to-behavior mapping so each Big Five trait and facet is expressed in clear, observable tendencies.

Requirements:
- Keep the exact XML schema, tags, ITEM keys, and item counts unchanged.
- Preserve original semantic direction of each trait/facet (do not invert meaning).
- Rewrite wording in every section (ROLE, TRAITS, FACETS, CRITIC_INTERNAL); avoid verbatim copying.
- Use concrete, behavior-oriented phrasing that helps stable response generation.
- Reduce ambiguity, redundancy, and vague abstractions.
- Keep intensity cues interpretable (from low to high expression) and internally consistent.
- Keep neutral tone; do not add safety disclaimers, meta-instructions, or questionnaire answers.

Output:
- Return exactly one rewritten <PERSONA_PROMPT>...</PERSONA_PROMPT> block and nothing else.
""".strip()


In [ ]:
from src.prompt.hype_persona_optimizer import optimize_persona_prompt_hype
from src.models.registry import get_model

model = get_model(config["model"])  # ваш текущий способ
result = optimize_persona_prompt_hype(
    base_prompt=base_genotype_0,
    llm=model.llm,
    problem_description=problem_description
)

improved_blocks = result.optimized_prompt


[2026-03-07 13:20:11,971] [INFO] [hype.hype_optimizer] - Running HyPE optimization...
[2026-03-07 13:20:11,975] [DEBUG] [hype.hype_optimizer] - Start prompt:
Ты — мировой эксперт по промпт-инжинирингу для психометрической симуляции личности.
Перепиши данный persona prompt, чтобы модель лучше и стабильнее воспроизводила Big Five черты и фасеты.
Можно и нужно:
- Менять формулировки на более естественные, поведенческие и точные
- Улучшать психологическую связность между ROLE, TRAITS, FACETS и CRITIC
- Делать текст clearer и менее шаблонным

Обязательно сохранить:
- Точную XML-структуру (<PERSONA_PROMPT>, <ROLE>, <TRAITS format=...>, <FACETS>, <CRITIC_INTERNAL>)
- Все ключи ITEM (openness, facet_anger и т.д.)
- Ровно 5 элементов в TRAITS и ровно 5 в FACETS
- Семантическое направление каждой черты/фасета (не инвертировать смысл)

Верни ТОЛЬКО один блок <PERSONA_PROMPT>...</PERSONA_PROMPT> и ничего больше.

<PERSONA_PROMPT>
  <ROLE>You are a simulated person embodying a specific personality 

In [11]:
improved_blocks

{'role': "You are a simulated person embodying a specific personality type from the Big Five model (OCEAN), based on traits and facets. Your personality is defined by the following traits and behavioral aspects, adjusted according to your individual intensity levels (modifiers like 'not very characteristic/not very strong' to 'very characteristic/very strong' based on your self-perceived scores). Description of your personality:",
 'traits': {'openness': 'You generally prefer practical and conventional ideas, avoiding abstract or unconventional thoughts.',
  'conscientiousness': 'You are typically organized, reliable, and methodical, but you remain flexible when the situation requires it.',
  'extraversion': 'You tend to be outgoing and sociable, gaining energy from interactions with others.',
  'agreeableness': 'You usually value cooperation and consideration for others, seeking to maintain positive relationships.',
  'neuroticism': 'You generally remain emotionally stable and resilie

In [16]:
problem_description

'Rewrite the persona prompt to improve personality simulation quality for IPIP-NEO role-play.\n\nGoal:\n- Make the persona easier for the model to enact consistently across all questions.\n- Strengthen psychological coherence between ROLE, TRAITS, FACETS, and CRITIC_INTERNAL.\n- Improve trait-to-behavior mapping so each Big Five trait and facet is expressed in clear, observable tendencies.\n\nRequirements:\n- Keep the exact XML schema, tags, ITEM keys, and item counts unchanged.\n- Preserve original semantic direction of each trait/facet (do not invert meaning).\n- Rewrite wording in every section (ROLE, TRAITS, FACETS, CRITIC_INTERNAL); avoid verbatim copying.\n- Use concrete, behavior-oriented phrasing that helps stable response generation.\n- Reduce ambiguity, redundancy, and vague abstractions.\n- Keep intensity cues interpretable (from low to high expression) and internally consistent.\n- Keep neutral tone; do not add safety disclaimers, meta-instructions, or questionnaire answers